# 02 · Receiver characterization

**Question:** what does the HackRF add *by itself*, with no signal coming in?

Every receiver shows things that are not out there: a noise floor, a spike at its own LO (DC), uneven gain across the span, spurs it makes itself, and mirror images of real signals. If we don't know them, the detector will "find" them and the classifier will try to name them.

**Setup:** nothing on the antenna port: best a **50 Ω load**, otherwise the antenna removed (a preview: an open port still picks up a little stray RF). Amp off, antenna power off. Close the Tayf dashboard first: only one program can hold the HackRF.

**What we measure**
1. **Floor vs gain:** how each gain stage (LNA, VGA) moves the floor → the gain-staging rule.
2. **Floor shape across the span:** filter edges and the DC spike, per sample rate → what the detector divides out.
3. **Spur map 1 MHz – 6 GHz:** signals the HackRF makes itself.
4. **Carrier check:** is a suspicious carrier fixed in frequency, or tied to the LO?
5. **Image rejection** (needs a real signal, so the antenna goes back on): how strongly signals appear mirrored around the LO.

The results are saved as the **receiver profile** (`data/receiver/profile.json`). Run the whole notebook top to bottom (*Run All*); change the parameters below to explore.

In [ ]:
# ---- Parameters: change these ----
CONDITION = "antenna removed"        # or "50 ohm load"
FREQS = [100e6, 433e6, 915e6, 1250e6, 2450e6, 5800e6]   # where to measure floor vs gain
LNA_LIST = [0, 8, 16, 24, 32, 40]
VGA_LIST = [0, 10, 20, 30, 40, 50, 62]
FLOOR_FS = 10e6                       # sample rate for floor vs gain

SHAPE_FC = 1250e6                     # a quiet frequency for the span shape
SHAPE_GAINS = (False, 32, 40)         # amp, LNA, VGA: receiver noise well above the ADC floor
RATES = [2e6, 4e6, 8e6, 10e6, 12.5e6, 16e6, 20e6]

SPUR_RANGE = (1e6, 6000e6)
SPUR_BIN = 100e3
SPUR_GAINS = (False, 24, 30)          # the scan presets' gain
SPUR_SECONDS = 10

# Carriers to check: the band scan's suspects (2026-10-29) and the strongest spur-map entries
CARRIERS = [40e6, 80e6, 120e6, 160e6, 200e6, 2200e6, 14e6,
            122.47e6, 160.03e6, 541.35e6, 1228e6, 3029e6, 5162e6]
SAVE = True

In [ ]:
import numpy as np
import plotly.graph_objects as go
from tayf.sources.radio import Gains
from tayf.receiver import characterize as C
from tayf.receiver.profile import save_profile, load_profile
from pathlib import Path

# Save into the project's data folder wherever the notebook is started from
DATA = Path(C.__file__).resolve().parents[2] / "data" / "receiver"
rx = C.Rx()
profile = {"condition_note": CONDITION}
x, _ = C.capture(rx, FREQS[0], FLOOR_FS, Gains(False, 16, 20), 100_000)
print("HackRF open, serial", rx.radio.serial)
profile["hackrf_serial"] = rx.radio.serial

## 1 · Floor vs gain: the gain-staging rule

With no input, the floor is made by three noise sources along the chain, added as powers:
- **ADC floor** (quantization of the 8-bit converter): fixed in dBFS, gain cannot move it.
- **VGA-stage noise**: enters before the VGA, so only VGA gain raises it.
- **Front-end noise**: enters before the LNA, so LNA + VGA raise it.

This is the Friis idea: the stage with enough gain in front of the others sets the noise. For weak signals you want the **front end** to dominate (enough LNA), and everything above the **ADC floor** (enough VGA). Beyond that, more gain just lifts signal and noise together, and only brings clipping closer.

In [ ]:
rows, fits = [], {}
for fc in FREQS:
    r = C.floor_vs_gain(rx, fc, FLOOR_FS, LNA_LIST, VGA_LIST)
    rows += r
    fits[fc] = C.fit_floor(r)
profile["floor_vs_gain"] = {"fs_hz": FLOOR_FS, "rows": rows, "fits": {str(int(k)): v for k, v in fits.items()}}

print(f"{'freq MHz':>9} {'ADC floor':>10} {'VGA noise':>10} {'front noise':>11} {'LNA where front wins':>21} {'VGA to clear ADC @LNA32':>24} {'fit err':>8}")
for fc, f in fits.items():
    print(f"{fc/1e6:>9.0f} {f['adc_floor_dbfs_hz']:>10.1f} {f['n_vga_dbfs_hz']:>10.1f} {f['n_front_dbfs_hz']:>11.1f} "
          f"{f['lna_db_front_dominates']:>18.1f} dB {f['vga_db_clears_adc_at_lna32']:>21.1f} dB {f['rms_error_db']:>6.2f} dB")
print("(noise levels in dBFS/Hz, the two sources referred to 0 dB gain)")

In [ ]:
fc = FREQS[3] if len(FREQS) > 3 else FREQS[0]       # one frequency in detail; change to see another
fig = go.Figure()
f = fits[fc]
vg = np.linspace(0, 62, 63)
for lna in LNA_LIST:
    pts = [r for r in rows if r["fc_hz"] == fc and r["lna_db"] == lna]
    fig.add_scatter(x=[r["vga_db"] for r in pts], y=[r["floor_dbfs_hz"] for r in pts], mode="markers", name=f"LNA {lna}")
    fig.add_scatter(x=vg, y=C.floor_model(lna, vg, f["adc_floor_dbfs_hz"], f["n_vga_dbfs_hz"], f["n_front_dbfs_hz"]),
                    mode="lines", line=dict(width=1, dash="dot"), showlegend=False, hoverinfo="skip")
fig.add_hline(y=f["adc_floor_dbfs_hz"], line_dash="dash", annotation_text="ADC floor")
fig.update_layout(height=420, title=f"Noise floor vs VGA gain at {fc/1e6:.0f} MHz (dots: measured, lines: three-source model)",
                  xaxis_title="VGA gain (dB)", yaxis_title="floor (dBFS/Hz)", margin=dict(t=50))
fig.show()

In [ ]:
# ADC use: how many 8-bit steps the noise covers, and where clipping starts
fig = go.Figure()
for lna in LNA_LIST:
    pts = [r for r in rows if r["fc_hz"] == fc and r["lna_db"] == lna]
    fig.add_scatter(x=[r["vga_db"] for r in pts], y=[r["rms_lsb"] for r in pts], name=f"LNA {lna}")
fig.update_layout(height=320, title="Noise size in ADC steps (rms, in LSB): below ~1 the converter barely sees it",
                  xaxis_title="VGA gain (dB)", yaxis_title="rms (LSB)", yaxis_type="log", margin=dict(t=50))
fig.show()
clips = [r for r in rows if r["clip_pct"] > 0.1]
print("Settings that clip on noise alone (> 0.1 %):", sorted({(r['lna_db'], r['vga_db']) for r in clips}) or "none")

**Read it:**
- At low VGA every curve sits on the ADC floor: the receiver noise is smaller than one ADC step, so it is invisible, and so is any signal weaker than it.
- Raising VGA lifts the floor once the receiver's noise passes the ADC floor.
- Curves for different LNA settings stay together until the LNA is large enough for the front end to dominate. Below that point, LNA gain does not improve sensitivity.
- **Rule:** LNA up to about the "front wins" value (if no strong signals overload it), then VGA until the floor is clearly above the ADC floor. More than that only costs headroom.

## 2 · Floor shape across the span

The same noise, averaged for ~2 M samples at each sample rate, shown relative to the middle of the span. We look at:
- **edges:** the anti-alias filter (HackRF sets it to 0.75 × fs, and it cannot go below 1.75 MHz),
- **DC spike** at the centre, before and after DC removal.

The detector (Phase 3, FCME) assumes a flat floor, so it will divide by this shape first.

In [ ]:
shapes = []
for fs in RATES:
    s = C.span_shape(rx, SHAPE_FC, fs, Gains(*SHAPE_GAINS))
    shapes.append(s)
profile["span_shape"] = shapes

fig = go.Figure()
for s in shapes:
    f, shape, raw = s["_full"]
    fig.add_scatter(x=f / s["fs_hz"], y=shape, name=f"{s['fs_hz']/1e6:g} MS/s", line=dict(width=1))
fig.update_layout(height=400, title="Floor shape (DC removed), relative to the middle of the span",
                  xaxis_title="offset / sample rate", yaxis_title="dB", yaxis_range=[-12, 6], margin=dict(t=50))
fig.show()

print(f"{'rate':>8} {'DC spike raw':>13} {'after removal':>14} {'edge @0.40':>11} {'edge @0.45':>11} {'flat ±1 dB':>11}")
for s in shapes:
    print(f"{s['fs_hz']/1e6:>6.1f} M {s['dc_spike_raw_db']:>10.1f} dB {s['dc_spike_removed_db']:>11.1f} dB "
          f"{s['edge_db_at_0p40']:>8.1f} dB {s['edge_db_at_0p45']:>8.1f} dB {100*s['flat_1db_frac']:>9.0f} %")

In [ ]:
# Zoom on the DC spike (one rate)
s = shapes[0]
f, shape, raw = s["_full"]
m = np.abs(f) < 0.05 * s["fs_hz"]
fig = go.Figure()
fig.add_scatter(x=f[m] / 1e3, y=raw[m], name="raw")
fig.add_scatter(x=f[m] / 1e3, y=shape[m], name="DC removed")
fig.update_layout(height=320, title=f"DC spike at {s['fs_hz']/1e6:g} MS/s", xaxis_title="offset (kHz)", yaxis_title="dB", margin=dict(t=50))
fig.show()

## 3 · Spur map, 1 MHz – 6 GHz

A whole-range `hackrf_sweep`, averaged for a few seconds. With no input, a narrow peak well above the local floor is a **spur**: made by the HackRF (clock harmonics, mixer products) or leaking in. With the antenna removed some strong local signals (FM, cellular) may still leak in through the open port; the 50 Ω run tells them apart.

In [ ]:
rx.close()                                  # hackrf_sweep needs the HackRF to itself
spurs = C.spur_map(*SPUR_RANGE, bin_hz=SPUR_BIN, gains=Gains(*SPUR_GAINS), seconds=SPUR_SECONDS)
profile["spurs"] = spurs
f, p, local = spurs["_full"]
fig = go.Figure()
fig.add_scattergl(x=f / 1e6, y=p, name="mean power", line=dict(width=1))
fig.add_scattergl(x=f / 1e6, y=local, name="local floor", line=dict(width=1, dash="dot"))
top = spurs["spurs"][:40]
fig.add_scattergl(x=[s["freq_hz"] / 1e6 for s in top], y=[np.interp(s["freq_hz"], f, p) for s in top],
                  mode="markers", marker=dict(size=7, symbol="x"), name="spurs (top 40)")
fig.update_layout(height=420, title=f"Spur map ({spurs['sweeps']} sweeps averaged)", xaxis_title="MHz", yaxis_title="dB / bin", margin=dict(t=50))
fig.show()
print(f"{len(spurs['spurs'])} spurs ≥ {spurs['threshold_db']} dB above the local floor. Strongest 25:")
for s in spurs["spurs"][:25]:
    print(f"  {s['freq_hz']/1e6:>10.3f} MHz  +{s['height_db']:>5.1f} dB  ({s['width_hz']/1e3:.0f} kHz wide)")

In [ ]:
# Are they on a pattern? Distance of each spur from the nearest multiple of 10 MHz
# (the HackRF's reference clock) and of 20 MHz (hackrf_sweep's tuning step).
for s in spurs["spurs"][:25]:
    fz = s["freq_hz"]
    print(f"{fz/1e6:>10.3f} MHz   off a 10 MHz multiple by {((fz + 5e6) % 10e6 - 5e6)/1e3:>7.0f} kHz")

## 4 · Carrier check

`hackrf_sweep` adds artifacts of its own (it hops and mixes fast), so not every spur in the map exists in live mode. For each carrier on the list: tune the LO 300 kHz below and 300 kHz above. A carrier that stays at the same **absolute** frequency in both is really there (a real transmitter, or a clock harmonic inside the HackRF); one that moves with the LO is an artifact of the receiver. With nothing on the input, "really there" points to the HackRF itself (or leakage).

In [ ]:
checks = [C.check_carrier(rx, fz, Gains(*SPUR_GAINS)) for fz in CARRIERS]
profile["carrier_checks"] = checks
for c in checks:
    a, b = c["captures"]
    seen = [x["above_floor_db"] > 6 for x in c["captures"]]
    verdict = ("fixed frequency: made inside the HackRF (or leaking in)" if c["fixed_frequency"] else
               "only at some LO settings: a tuning-dependent receiver spur" if any(seen) else
               "absent with nothing connected: a real outside signal if a scan saw it with the antenna, "
               "a hackrf_sweep artifact if only the spur map saw it")
    print(f"{c['f_hz']/1e6:>9.3f} MHz: {a['above_floor_db']:>5.1f} dB / {b['above_floor_db']:>5.1f} dB above floor → {verdict}")
# Carriers the HackRF makes itself, at a fixed frequency: live mode will see these too
profile["internal_carriers"] = [{"freq_hz": c["captures"][0]["peak_hz"], "above_floor_db": c["captures"][0]["above_floor_db"],
                                 "gains": list(SPUR_GAINS)} for c in checks if c["fixed_frequency"]]
print("Internal carriers:", [f"{c['freq_hz']/1e6:.3f} MHz" for c in profile["internal_carriers"]])

## 5 · Save the profile

In [ ]:
rx.close()
if SAVE:
    path = save_profile(profile, CONDITION, DATA)
    print("Saved", path)
    print("and", DATA / "profile.json")

## 6 · Image rejection (antenna back on)

The HackRF mixes the signal down with two paths, I and Q. If they are not perfectly balanced, every signal also appears **mirrored** around the LO, weaker by the **image rejection ratio (IRR)**.

There is no signal generator, so a strong FM station is the test signal. The method is in `C.image_rejection`: measure at the mirror position, then move the LO so the image moves away, and subtract what is really there.

**To run:** screw the whip back on, set `RUN_IMAGE_TEST = True`, and run the two cells below. Pick a strong station from the FM scan.

In [ ]:
RUN_IMAGE_TEST = False
F_STATION = 95.3e6                       # a strong FM station
OFFSETS = [0.3e6, 1e6, 2e6, 3e6]          # where the station sits relative to the LO
IMAGE_GAINS = (False, 24, 20)             # keep well below clipping: 10 MS/s holds many stations

In [ ]:
if RUN_IMAGE_TEST:
    rx = C.Rx()
    img = C.image_rejection(rx, F_STATION, OFFSETS, Gains(*IMAGE_GAINS))
    rx.close()
    print(f"{'offset':>8} {'station':>9} {'mirror A':>9} {'mirror B':>9} {'IRR':>10}")
    for r in img:
        irr = f"≥ {r['irr_db']:.1f}" if r["irr_is_lower_bound"] else f"{r['irr_db']:.1f}"
        print(f"{r['offset_hz']/1e6:>6.1f} M {r['signal_db']:>7.1f} dB {r['mirror_A_db']:>7.1f} dB {r['mirror_B_db']:>7.1f} dB {irr:>7} dB")
    print("Station level vs its offset from the LO (the 97.9 MHz question):")
    for r in img:
        print(f"  at +{r['offset_hz']/1e6:.1f} MHz: {r['signal_db']:.1f} dB    at +{r['b_offset_hz']/1e6:.1f} MHz: {r['signal_db_at_B_offset']:.1f} dB")
    if SAVE:
        prof = load_profile(DATA)
        prof["image_rejection"] = {"f_station_hz": F_STATION, "gains": list(IMAGE_GAINS), "rows": img}
        cond = prof.pop("condition"); prof.pop("created", None)
        print("Updated", save_profile(prof, cond, DATA))

## Try it yourself
- Change `fc` in section 1 to another frequency: is the front end equally quiet at 5.8 GHz?
- Re-run with `CONDITION = "50 ohm load"` once the terminator is on, and compare the spur lists: spurs that disappear were leaking in through the open port.
- Turn the amp on (`Gains(True, ...)`) in section 1: how many dB does it lower the front-end noise relative to the signal?